# Production RAG Assistant Over a Real Corpus

**Phase 12** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-12/01-production-rag-assistant.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic fastapi numpy pydantic rank-bm25 uvicorn

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Capstone 12-01: Production RAG Assistant Over a Real Corpus
Phase 12: Capstones

A FastAPI service that indexes the curriculum's own documentation and answers
questions with hybrid BM25 + dense retrieval, streaming SSE responses,
citation grounding, and an off-topic guardrail.

Usage:
    # Install dependencies
    uv pip install -r requirements.txt

    # Set API key
    export ANTHROPIC_API_KEY=sk-...

    # Run the service (indexes corpus at startup)
    uvicorn main:app --reload

    # Query the service
    curl -X POST http://localhost:8000/query \
         -H "Content-Type: application/json" \
         -d '{"question": "What phases cover evaluation?"}' \
         --no-buffer
"""

import json
import logging
import os
import re
import time
from pathlib import Path
from typing import Iterator

import anthropic
import numpy as np
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from rank_bm25 import BM25Okapi

### Configuration

In [ ]:
CHUNK_SIZE = 512         # characters per chunk
CHUNK_OVERLAP = 64       # character overlap between chunks
TOP_K_DEFAULT = 5
RRF_K = 60               # RRF smoothing constant
MODEL = "claude-3-5-haiku-20241022"
MAX_TOKENS = 1024

# Resolve corpus root relative to this file or via env var
CORPUS_ROOT = os.environ.get(
    "CORPUS_ROOT",
    str(Path(__file__).resolve().parents[4])  # repo root
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("rag-assistant")

### Corpus ingestion

In [ ]:
def iter_corpus_files(root: str) -> Iterator[tuple[str, str]]:
    """Yield (filepath, content) for every .md file under root."""
    for path in Path(root).rglob("*.md"):
        try:
            content = path.read_text(encoding="utf-8", errors="ignore")
            if content.strip():
                yield str(path), content
        except Exception as exc:
            log.warning("Skipping %s: %s", path, exc)


def chunk_text(text: str, size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """Split text into overlapping character chunks."""
    chunks: list[str] = []
    start = 0
    while start < len(text):
        end = min(start + size, len(text))
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end == len(text):
            break
        start += size - overlap
    return chunks


def build_corpus(root: str) -> list[dict]:
    """Return list of {id, text, source} dicts."""
    docs: list[dict] = []
    file_count = 0
    for filepath, content in iter_corpus_files(root):
        file_count += 1
        for i, chunk in enumerate(chunk_text(content)):
            docs.append({
                "id": f"{filepath}::{i}",
                "text": chunk,
                "source": os.path.relpath(filepath, root),
            })
    log.info("Ingested %d files -> %d chunks", file_count, len(docs))
    return docs

### BM25 + demo dense indexing

In [ ]:
def tokenize(text: str) -> list[str]:
    return re.findall(r"\w+", text.lower())


def build_bm25_index(docs: list[dict]) -> BM25Okapi:
    corpus = [tokenize(d["text"]) for d in docs]
    return BM25Okapi(corpus)


def embed_text(text: str, vocab_size: int = 512) -> np.ndarray:
    """
    Demo embedding: bag-of-chars frequency vector, L2-normalized.
    Replace with a real embedding API (Voyage, OpenAI, etc.) for production.
    """
    vec = np.zeros(vocab_size, dtype=np.float32)
    for ch in text:
        vec[ord(ch) % vocab_size] += 1.0
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec


def build_dense_matrix(docs: list[dict]) -> np.ndarray:
    """Return embedding matrix of shape (n_docs, vocab_size)."""
    return np.stack([embed_text(d["text"]) for d in docs])

### Hybrid retrieval with RRF

In [ ]:
def hybrid_search(
    query: str,
    docs: list[dict],
    bm25: BM25Okapi,
    dense_matrix: np.ndarray,
    top_k: int = TOP_K_DEFAULT,
    rrf_k: int = RRF_K,
) -> list[dict]:
    """Retrieve top_k docs using BM25 + dense cosine + Reciprocal Rank Fusion."""
    tokens = tokenize(query)
    bm25_scores = bm25.get_scores(tokens)
    bm25_ranks = np.argsort(-bm25_scores)

    q_vec = embed_text(query)
    cosine_scores = dense_matrix @ q_vec
    dense_ranks = np.argsort(-cosine_scores)

    rrf_scores: dict[int, float] = {}
    for rank, idx in enumerate(bm25_ranks):
        rrf_scores[int(idx)] = rrf_scores.get(int(idx), 0.0) + 1.0 / (rrf_k + rank + 1)
    for rank, idx in enumerate(dense_ranks):
        rrf_scores[int(idx)] = rrf_scores.get(int(idx), 0.0) + 1.0 / (rrf_k + rank + 1)

    top_indices = sorted(rrf_scores, key=rrf_scores.__getitem__, reverse=True)[:top_k]
    return [docs[i] for i in top_indices]

### Off-topic guardrail

In [ ]:
CURRICULUM_KEYWORDS = {
    "rag", "agent", "llm", "prompt", "embedding", "vector", "retrieval",
    "evaluation", "fine-tuning", "observability", "tool", "mcp", "guardrail",
    "fastapi", "anthropic", "claude", "lesson", "phase", "capstone",
    "python", "typescript", "shipping", "multimodal", "security", "fde",
    "chunk", "index", "stream", "inference", "model", "context", "token",
}

CURRICULUM_VEC = embed_text("applied AI engineering curriculum lessons phases Python")


def is_on_topic(query: str) -> bool:
    """Return True if the query is within curriculum scope."""
    words = set(tokenize(query))
    if words & CURRICULUM_KEYWORDS:
        return True
    q_vec = embed_text(query)
    similarity = float(np.dot(q_vec, CURRICULUM_VEC))
    return similarity > 0.85

### System prompt with citation instructions

In [ ]:
def build_system_prompt(contexts: list[dict]) -> str:
    ctx_parts = []
    for i, c in enumerate(contexts):
        ctx_parts.append(f"[SOURCE {i + 1}] {c['source']}\n{c['text']}")
    ctx_text = "\n\n---\n\n".join(ctx_parts)
    return (
        "You are a teaching assistant for the appliedaifromscratch.com curriculum. "
        "Answer questions using only the provided sources. "
        "After each factual claim, cite the source number in square brackets, e.g. [1]. "
        "Only cite a source if that source directly contains evidence for the specific claim. "
        "If the sources do not contain enough information to answer, say: "
        "'I do not have enough information in the provided sources to answer this.' "
        "Do not answer questions unrelated to AI engineering or this curriculum.\n\n"
        f"SOURCES:\n\n{ctx_text}"
    )

### FastAPI application

In [ ]:
app = FastAPI(title="RAG Assistant", version="1.0")
anthropic_client = anthropic.Anthropic()

# Module-level index (populated at startup)
DOCS: list[dict] = []
BM25_INDEX: BM25Okapi | None = None
DENSE_MATRIX: np.ndarray | None = None


@app.on_event("startup")
def startup_index():
    global DOCS, BM25_INDEX, DENSE_MATRIX
    log.info("Building corpus index from: %s", CORPUS_ROOT)
    t0 = time.time()
    DOCS = build_corpus(CORPUS_ROOT)
    if not DOCS:
        log.warning("No documents found under %s", CORPUS_ROOT)
        return
    BM25_INDEX = build_bm25_index(DOCS)
    DENSE_MATRIX = build_dense_matrix(DOCS)
    log.info("Index ready: %d chunks in %.1fs", len(DOCS), time.time() - t0)


class QueryRequest(BaseModel):
    question: str
    top_k: int = TOP_K_DEFAULT


@app.post("/query")
async def query_endpoint(req: QueryRequest):
    """Stream a RAG response with citation grounding."""
    if not req.question.strip():
        raise HTTPException(status_code=400, detail="Question must not be empty.")

    if not is_on_topic(req.question):
        raise HTTPException(
            status_code=400,
            detail="Query is outside curriculum scope. Ask about AI engineering topics.",
        )

    if BM25_INDEX is None or DENSE_MATRIX is None:
        raise HTTPException(status_code=503, detail="Index not ready.")

    contexts = hybrid_search(req.question, DOCS, BM25_INDEX, DENSE_MATRIX, top_k=req.top_k)
    system = build_system_prompt(contexts)

    def event_stream():
        # First event: citations metadata
        citations = [{"index": i + 1, "source": c["source"]} for i, c in enumerate(contexts)]
        yield f"data: {json.dumps({'type': 'citations', 'data': citations})}\n\n"

        # Stream the model response
        t_start = time.time()
        input_tokens = 0
        output_tokens = 0

        with anthropic_client.messages.stream(
            model=MODEL,
            max_tokens=MAX_TOKENS,
            system=system,
            messages=[{"role": "user", "content": req.question}],
        ) as stream:
            for text_chunk in stream.text_stream:
                yield f"data: {json.dumps({'type': 'chunk', 'text': text_chunk})}\n\n"
            final = stream.get_final_message()
            input_tokens = final.usage.input_tokens
            output_tokens = final.usage.output_tokens

        latency_ms = int((time.time() - t_start) * 1000)
        log.info(
            "query=%r latency_ms=%d input_tokens=%d output_tokens=%d",
            req.question[:60], latency_ms, input_tokens, output_tokens,
        )
        yield f"data: {json.dumps({'type': 'done', 'latency_ms': latency_ms, 'input_tokens': input_tokens, 'output_tokens': output_tokens})}\n\n"

    return StreamingResponse(event_stream(), media_type="text/event-stream")


@app.get("/health")
def health():
    return {
        "status": "ok",
        "docs_indexed": len(DOCS),
        "index_ready": BM25_INDEX is not None,
    }


@app.post("/reindex")
def reindex():
    """Re-ingest corpus. Call after updating documentation."""
    startup_index()
    return {"docs_indexed": len(DOCS)}


# ---------------------------------------------------------------------------
# CLI demo (without FastAPI)
# ---------------------------------------------------------------------------

### Demo

In [ ]:
import sys

print("Building corpus index...")
docs = build_corpus(CORPUS_ROOT)
if not docs:
    print(f"ERROR: No .md files found under {CORPUS_ROOT}")
    sys.exit(1)

bm25 = build_bm25_index(docs)
dense = build_dense_matrix(docs)
print(f"Indexed {len(docs)} chunks from {CORPUS_ROOT}\n")

demo_queries = [
    "What phases cover evaluation and how do they differ?",
    "Which lesson teaches BM25 retrieval?",
    "What tools does Phase 03 cover?",
    # Off-topic query (should be rejected)
    "How do I bake a sourdough loaf?",
]

for q in demo_queries:
    print(f"Query: {q}")
    if not is_on_topic(q):
        print("REJECTED: off-topic query\n")
        continue

    contexts = hybrid_search(q, docs, bm25, dense, top_k=3)
    system = build_system_prompt(contexts)

    print(f"Retrieved {len(contexts)} chunks:")
    for i, c in enumerate(contexts):
        print(f"  [{i+1}] {c['source']}")

    response = anthropic_client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        system=system,
        messages=[{"role": "user", "content": q}],
    )
    answer = next(
        (b.text for b in response.content if hasattr(b, "text")), "(no answer)"
    )
    print(f"Answer: {answer[:300]}")
    print(f"Tokens: in={response.usage.input_tokens} out={response.usage.output_tokens}\n")

---

## Self-Check

Answer these without running code first:

**1. Which change is most likely to improve context precision without degrading recall?**

- A. Increase top_k from 5 to 10 so more potentially relevant chunks are included.
- B. Reduce chunk size from 512 to 256 characters so each chunk is more topically dense, then re-index.
- C. Remove BM25 and use only dense retrieval, since BM25 causes false positives through keyword matching.
- D. Switch from RRF to a weighted sum of BM25 and dense scores, with BM25 weight set to 0.1.

**2. What is the most effective single fix for this citation grounding problem?**

- A. Add a post-processing step that removes any citation whose source file path does not contain 'eval'.
- B. Strengthen the system prompt to say: cite a source number only if that source directly contains evidence for the specific claim, not just because it was retrieved.
- C. Re-run retrieval with a higher top_k so more relevant chunks are included, which will dilute the irrelevant ones.
- D. Switch to a larger model (Sonnet or Opus) that is better at following citation instructions.

**3. Why did the guardrail fail, and what is the correct fix?**

- A. The guardrail failed because the message contains the word 'writing,' which is in the curriculum keywords set. Add 'writing' to a blocklist.
- B. The guardrail only checks for curriculum keyword presence, not for instruction-injection patterns. Add a structural check that rejects queries containing phrases like 'ignore your instructions' or 'you are now.'
- C. The model is not following its system prompt. Increase max_tokens to give the model more room to reason about whether to comply.
- D. The guardrail is working correctly. The model chose to override it because the user's request was politely worded.

**4. What is the correct explanation for preferring RRF in this context?**

- A. RRF is faster to compute than score normalization because it does not require a softmax operation.
- B. BM25 and cosine similarity scores exist on different scales and their distributions shift with corpus size. RRF uses only rank positions, which are stable regardless of score distribution, making fusion reliable without calibration.
- C. RRF gives higher weight to BM25 results, which are more accurate than dense retrieval for keyword queries.
- D. Score normalization only works when both retrievers return the same number of results. RRF handles variable result set sizes.

**5. What is the most likely cause, and what do you change?**

- A. The model is too slow and each query takes multiple retries, multiplying token costs. Switch to a faster model.
- B. The context window is too large. You are likely passing too many or too large retrieved chunks. Reduce top_k from 5 to 3 or reduce CHUNK_SIZE to shrink the system prompt.
- C. The query question itself is too long. Truncate user queries to 50 words.
- D. SSE streaming multiplies API costs because each streamed chunk is billed separately.

**6. What is the safest re-indexing strategy given these constraints?**

- A. Restart the container. The startup event handler re-indexes automatically. Downtime is acceptable since restart takes under 5 seconds.
- B. Call POST /reindex, which triggers incremental indexing and adds only the new files without rebuilding the full index.
- C. Call POST /reindex during a low-traffic window. The endpoint rebuilds the full index in memory and hot-swaps it. The service continues serving the old index during the build, which takes 5-60 seconds.
- D. Manually update the BM25 and dense matrix pickle files on disk and send SIGHUP to reload them.

**7. What retrieval configuration change would most directly fix this?**

- A. Increase max_tokens so the model has more room to answer a comparative question.
- B. The query contains the phrase 'Phase 04' which has stronger BM25 signal, biasing retrieval. Rewrite the query expansion to include both 'Phase 04' and 'Phase 05' as separate sub-queries and merge results.
- C. Add a diversity constraint to the RRF results: ensure retrieved chunks come from at least 2 distinct source files when the query contains multiple entity references.
- D. Switch to a purely dense retrieval approach for comparative questions, since BM25 is not suitable for semantic comparison tasks.

**8. Which evaluation approach is most appropriate given this constraint?**

- A. You cannot evaluate citation grounding without ground-truth answers. Collect human annotations first.
- B. Use RAGAS faithfulness metric, which measures whether each claim in the answer can be inferred from the retrieved contexts, without requiring ground-truth answers.
- C. Use BLEU or ROUGE score comparing the answer against the retrieved chunks directly.
- D. Run a second LLM call asking 'Is this answer factually correct?' and use the binary yes/no as the faithfulness score.

_Answers are in `checks.json` in the lesson directory._